In [1]:
!pip3 install -U torch datasets transformers==4.35.2 bitsandbytes peft==0.5.0 accelerate==0.24.1

In [2]:
import torch

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, BitsAndBytesConfig

from peft import LoraConfig, get_peft_model, TaskType


/home/dk/.local/lib/python3.12/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/dk/.local/lib/python3.12/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [3]:
model_name = 'TinyLLama/TinyLlama-1.1B-Chat-v1.0'

bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type = 'nf4',
    bnb_4bit_compute_dtype = torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map = 'auto',
    trust_remote_code = True
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)



/home/dk/.local/lib/python3.12/site-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [4]:
lora_config = LoraConfig(
    r = 8,
    lora_alpha = 16,
    target_modules = ['q_proj', 'v_proj'],
    lora_dropout = 0.05,
    bias = 'none',
    task_type = TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)

In [5]:
data = load_dataset('json', data_files='kubernetes.jsonl')['train']

In [6]:

def tokenize(batch):
    texts = [
        f"### Instruction:\n{inst}\n### Response:\n{out}"
        for inst, out in zip(batch['instruction'], batch['response'])
    ]

    tokens = tokenizer(
        texts,
        padding = 'max_length',
        truncation = True,
        max_length = 256,
        return_tensors = 'pt'
    )

    tokens['labels'] = tokens['input_ids'].clone()

    return tokens


In [7]:
tokenized_data = data.map(tokenize, batched=True, remove_columns=data.column_names)

In [8]:

training_args = TrainingArguments(
    output_dir = './tinyllama-lora-tuned-kubermetes',
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 2,
    learning_rate = 1e-3,
    num_train_epochs = 50,
    fp16 = True,
    logging_steps = 20,
    save_strategy = 'epoch',
    report_to = 'none',
    remove_unused_columns = False,
    label_names = ["labels"]
)

In [9]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data,
    tokenizer=tokenizer
)

/home/dk/.local/lib/python3.12/site-packages/accelerate/accelerator.py:439: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [10]:
trainer.train()


You're using a LlamaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss
20,4.274400
40,0.130100
60,0.018500
80,0.012000
100,0.011000
120,0.010600
140,0.010300


TrainOutput(global_step=150, training_loss=0.5962600088616212, metrics={'train_runtime': 111.4236, 'train_samples_per_second': 4.936, 'train_steps_per_second': 1.346, 'total_flos': 874907644723200.0, 'train_loss': 0.5962600088616212, 'epoch': 50.0})

In [11]:
model.save_pretrained("./tinyllama-lora-tuned-adapter-kubernetes")
tokenizer.save_pretrained("./tinyllama-lora-tuned-adapter-kubernetes")

('./tinyllama-lora-tuned-adapter-kubernetes/tokenizer_config.json',
 './tinyllama-lora-tuned-adapter-kubernetes/special_tokens_map.json',
 './tinyllama-lora-tuned-adapter-kubernetes/tokenizer.json')

In [12]:
from google.colab import drive
drive.mount('/content/drive')


ModuleNotFoundError: No module named 'google.colab'